# Regressão Logística

Nesse notebook nós iremos treinar um modelo de regressão logística como baseline. O objetivo aqui é checar um desempenho mínimo aceitável para nossa modelagem e identificar possíveis erros no nosso dataset (como data leakage e features irrelevantes).

## Importando as Bibliotecas

Primeiro vamos carregar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carregando o Dataset

Agora vamos carregar nosso dataset.

In [53]:
df = pd.read_parquet("../../data/processed/prepared/UCMF_simple_imputer.parquet")

## Treinando o Modelo

Agora vamos treinar nosso modelo.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import (
    SimpleImputer,
    KNNImputer,
    IterativeImputer,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
    TargetEncoder,
)
from sklearn.preprocessing import (
    MaxAbsScaler,
    StandardScaler,
    RobustScaler
)
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)
from sklearn.metrics import classification_report

random_state = 42

X = df.drop(["patologia"], axis="columns")
y = df["patologia"]

numeric_columns = X.select_dtypes(include="number").columns.to_list()
categorical_columns = X.select_dtypes(include="category").columns.to_list()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.7,
    random_state=random_state,
    stratify=y
)

cross_validator = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_state,
)

model = LogisticRegressionCV(
    Cs=list(np.linspace(1, 100, 10)),
    cv=cross_validator,
    max_iter=1000
)

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore"
)

scaler = MaxAbsScaler()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler")
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric_features", "", ""),
        ("categorical_features", "", "")
    ],
    remainder="passthrough"
)

pipeline = Pipeline(steps=[
    ("encoder", encoder),
    ("scaler", scaler),
    ("model", model),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of

              precision    recall  f1-score   support

     ausente       0.73      0.80      0.76      2024
    presente       0.68      0.59      0.63      1488

    accuracy                           0.71      3512
   macro avg       0.70      0.69      0.70      3512
weighted avg       0.71      0.71      0.71      3512



/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 15, 18, 19, 24] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
